# Transfer learning untuk resolusi alias unit

Notebook ini membuat proof-of-concept model ML untuk mencocokkan nama unit kontraktor yang bervariasi dengan canonical unit type. Model hanya memberi kandidat dan ranking; model tidak menghitung Fuel Ratio, tidak mengisi fuel rate yang hilang, dan tidak menggantikan review manusia.

Sumber data adalah workbook resmi dan aturan alias pada `docs/system-design/05-unit-alias-resolution.md`. Perhitungan Fuel Ratio tetap menjadi tanggung jawab calculation engine.

Di Google Colab, letakkan tiga file sumber langsung di panel Files pada `/content/`: `01_FUEL_RATIO_CALCULATION.xlsx`, `03_REFERENCE_FUEL_CONSUMPTION.md`, dan `05-unit-alias-resolution.md`.

In [1]:
%pip install -q --upgrade --no-deps "sentence-transformers==3.4.1" "transformers==4.46.3" "tokenizers==0.20.3" "huggingface-hub>=0.23,<1.0"
%pip install -q "pandas>=2" openpyxl scikit-learn matplotlib joblib
%env WANDB_DISABLED=true

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.9/275.9 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 86.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 85.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.3 MB/s eta 0:00:00
env: WANDB_DISABLED=true


Jika sebelumnya cell import pernah dijalankan dan gagal, jalankan cell instalasi ini lalu pilih **Runtime → Restart session**. Setelah restart, jalankan notebook dari awal agar versi package baru dimuat.

## 1. Konfigurasi dan provenance

Jalankan notebook dari `server/` atau dari direktori project. Pretrained encoder akan diunduh oleh Sentence-Transformers pada eksekusi pertama.

In [2]:
from __future__ import annotations

import json
import os
import platform
import random
import re
import sys
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import InputExample, SentenceTransformer, losses
from torch.utils.data import DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    torch.cuda.manual_seed_all(SEED)
print(f"Compute device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

cwd = Path.cwd().resolve()
server_root = next((p for p in (cwd, *cwd.parents) if p.name == "server"), None)
colab_files = Path("/content")
direct_colab_sources = (
    colab_files / "01_FUEL_RATIO_CALCULATION.xlsx",
    colab_files / "03_REFERENCE_FUEL_CONSUMPTION.md",
    colab_files / "05-unit-alias-resolution.md",
)
if server_root is None and not all(path.exists() for path in direct_colab_sources) and Path("/content/server").exists():
    server_root = Path("/content/server")
if server_root is None:
    server_root = Path("/content")
    project_root = server_root
    workbook_path = project_root / "01_FUEL_RATIO_CALCULATION.xlsx"
    reference_doc_path = project_root / "03_REFERENCE_FUEL_CONSUMPTION.md"
    alias_doc_path = project_root / "05-unit-alias-resolution.md"
else:
    project_root = server_root.parent
    workbook_path = project_root / "docs" / "01_FUEL_RATIO_CALCULATION.xlsx"
    reference_doc_path = project_root / "docs" / "03_REFERENCE_FUEL_CONSUMPTION.md"
    alias_doc_path = project_root / "docs" / "system-design" / "05-unit-alias-resolution.md"
for source_path in (workbook_path, reference_doc_path, alias_doc_path):
    if not source_path.exists():
        raise FileNotFoundError(source_path)

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
print({
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "workbook": str(workbook_path),
    "model": MODEL_NAME,
})

Compute device: cpu
{'python': '3.12.13', 'platform': 'Linux-6.6.122+-x86_64-with-glibc2.35', 'workbook': '/content/01_FUEL_RATIO_CALCULATION.xlsx', 'model': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'}


## 2. Normalisasi dan pemuatan workbook

Normalisasi mengikuti kontrak alias: NFKC, uppercase, collapse whitespace, dan penghapusan separator. Nama mentah tidak pernah dibuang.

In [3]:
def normalize_name(value: str) -> str:
    normalized = unicodedata.normalize("NFKC", str(value)).strip().upper()
    normalized = re.sub(r"\s+", " ", normalized)
    return re.sub(r"[\s\-_.\/]", "", normalized)


def activity_key(value: object) -> str | None:
    if pd.isna(value):
        return None
    return str(value).strip().lower()

equipment = pd.read_excel(workbook_path, sheet_name="Equipment Summary", header=2)
equipment = equipment.rename(
    columns={
        "Unit type list": "canonical_name",
        "Activity": "activity",
    }
)
equipment = equipment[["canonical_name", "activity"]].dropna(subset=["canonical_name"])
equipment["canonical_name"] = equipment["canonical_name"].astype(str).str.strip()
equipment["activity"] = equipment["activity"].map(activity_key)

dummy = pd.read_excel(workbook_path, sheet_name="DUMMY DATA", header=1)
dummy = dummy.rename(columns={"UNIT": "raw_name", "ACTIVITY": "activity"})
dummy = dummy[["raw_name", "activity"]].dropna(subset=["raw_name"])
dummy["raw_name"] = dummy["raw_name"].astype(str).str.strip()
dummy["activity"] = dummy["activity"].map(activity_key)

reference = pd.read_excel(workbook_path, sheet_name="Ref_fuel", header=2)
reference = reference.rename(columns={"TYPE": "reference_name", "ACTIVITY": "activity"})
reference = reference[["reference_name", "activity"]].dropna(subset=["reference_name"])
reference["reference_name"] = reference["reference_name"].astype(str).str.strip()
reference["activity"] = reference["activity"].map(activity_key)

print(f"Equipment Summary rows: {len(equipment)}")
print(f"DUMMY DATA rows: {len(dummy)}")
print(f"Ref_fuel rows: {len(reference)}")
display(equipment.head())
display(dummy.head())

Equipment Summary rows: 51
DUMMY DATA rows: 38
Ref_fuel rows: 1153


,canonical_name,activity
0,EX26007,loading
1,PC125011R,loading
2,PC1250SP8,loading
3,PC200011R,loading
4,PC20008,loading


,raw_name,activity
0,HT 2600,loading
1,PC 1250,loading
2,PC 1250_Mud,loading
3,PC 2000,loading
4,PC 2000_Mud,loading


## 3. Katalog canonical dan pasangan alias terverifikasi

Pasangan eksplisit berikut berasal dari contoh nyata pada dokumen desain alias. Variasi formatting dibuat sebagai augmentasi string saja; tidak ada nilai bisnis baru yang dibuat.

In [4]:
verified_alias_pairs = [
    ("FM9 ", "FM9"),
    ("RF-85MW", "RF85MW"),
    ("D155A-6R", "D155A6R"),
    ("D85ESS-2", "D85ESS2"),
    ("MF-420E", "MF420E"),
    ("mf420exhv", "MF420EXHV"),
    ("PUMP, MULTIFLO, MF420EXHV, CAT C27", "MF420EXHV"),
]

def canonical_variants(name: str) -> list[str]:
    variants = {name, name.upper(), name.lower()}
    variants.add(name.replace("-", " "))
    variants.add(name.replace("-", "_"))
    variants.add(name.replace(" ", "-"))
    return sorted(variant for variant in variants if variant.strip())

catalog = equipment[["canonical_name", "activity"]].copy()
catalog["normalized_name"] = catalog["canonical_name"].map(normalize_name)
catalog = catalog.drop_duplicates(subset=["normalized_name"], keep="first")

# Golden canonical names from the alias specification are included as candidates.
for _, canonical_name in verified_alias_pairs:
    if normalize_name(canonical_name) not in set(catalog["normalized_name"]):
        catalog.loc[len(catalog)] = {
            "canonical_name": canonical_name,
            "activity": None,
            "normalized_name": normalize_name(canonical_name),
        }

canonical_by_normalized = dict(zip(catalog["normalized_name"], catalog["canonical_name"]))
activity_by_normalized = dict(zip(catalog["normalized_name"], catalog["activity"]))

pairs: list[dict[str, str]] = []
def add_pair(raw_name: str, canonical_name: str, source: str) -> None:
    pairs.append({"raw_name": raw_name, "canonical_name": canonical_name, "source": source})

for raw_name, canonical_name in verified_alias_pairs:
    add_pair(raw_name, canonical_name, "verified_design_case")
    for variant in canonical_variants(raw_name):
        add_pair(variant, canonical_name, "verified_format_augmentation")

for row in dummy.itertuples(index=False):
    normalized = normalize_name(row.raw_name)
    if normalized in canonical_by_normalized:
        canonical_name = canonical_by_normalized[normalized]
        add_pair(row.raw_name, canonical_name, "deterministic_workbook_match")
        for variant in canonical_variants(row.raw_name):
            add_pair(variant, canonical_name, "workbook_format_augmentation")

pairs_df = pd.DataFrame(pairs).drop_duplicates(subset=["raw_name", "canonical_name"])
pairs_df["group"] = pairs_df["canonical_name"].map(normalize_name)
print(f"Canonical candidates: {len(catalog)}")
print(f"Training pairs: {len(pairs_df)}")
display(pairs_df.groupby("source").size().rename("pairs").to_frame())
display(pairs_df.head(12))

Canonical candidates: 47
Training pairs: 33


,pairs
source,
deterministic_workbook_match,3
verified_design_case,7
verified_format_augmentation,17
workbook_format_augmentation,6


,raw_name,canonical_name,source,group
0,FM9,FM9,verified_design_case,FM9
2,FM9-,FM9,verified_format_augmentation,FM9
3,fm9,FM9,verified_format_augmentation,FM9
4,RF-85MW,RF85MW,verified_design_case,RF85MW
5,RF 85MW,RF85MW,verified_format_augmentation,RF85MW
7,RF_85MW,RF85MW,verified_format_augmentation,RF85MW
8,rf-85mw,RF85MW,verified_format_augmentation,RF85MW
9,D155A-6R,D155A6R,verified_design_case,D155A6R
10,D155A 6R,D155A6R,verified_format_augmentation,D155A6R
12,D155A_6R,D155A6R,verified_format_augmentation,D155A6R


## 4. Fine-tuning encoder transfer learning

Model dasar adalah pretrained multilingual MiniLM. Fine-tuning memakai contrastive loss; validasi dipisahkan berdasarkan canonical unit agar variasi nama dari unit yang sama tidak bocor ke train dan validation.

In [5]:
if pairs_df["group"].nunique() >= 4:
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED)
    train_idx, validation_idx = next(splitter.split(pairs_df, groups=pairs_df["group"]))
    train_pairs = pairs_df.iloc[train_idx].reset_index(drop=True)
    validation_pairs = pairs_df.iloc[validation_idx].reset_index(drop=True)
else:
    train_pairs = pairs_df.copy()
    validation_pairs = pairs_df.copy()
    print("Peringatan: canonical group terlalu sedikit; evaluasi memakai pasangan yang sama dengan training.")

base_encoder = SentenceTransformer(MODEL_NAME, device=device)
base_encoder.max_seq_length = 64
encoder = SentenceTransformer(MODEL_NAME, device=device)
encoder.max_seq_length = 64
train_examples = [
    InputExample(texts=[row.raw_name, row.canonical_name])
    for row in train_pairs.itertuples(index=False)
]
train_loader = DataLoader(train_examples, shuffle=True, batch_size=min(16, max(2, len(train_examples))))
train_loss = losses.MultipleNegativesRankingLoss(encoder)

encoder.fit(
    train_objectives=[(train_loader, train_loss)],
    epochs=1,
    warmup_steps=max(1, len(train_loader) // 10),
    show_progress_bar=True,
)
print(f"Train pairs: {len(train_pairs)} | Validation pairs: {len(validation_pairs)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Train pairs: 21 | Validation pairs: 12


## 5. Retrieval dan evaluasi

Model digunakan untuk meranking kandidat, bukan untuk menerima mapping secara buta. Baseline dan model fine-tuned dibandingkan pada metrik retrieval.

In [6]:
canonical_names = catalog["canonical_name"].tolist()
canonical_embeddings = encoder.encode(
    canonical_names, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False
)

def retrieval_metrics(model: SentenceTransformer, examples: pd.DataFrame) -> dict[str, float]:
    embeddings = model.encode(
        examples["raw_name"].tolist(),
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    candidate_embeddings = model.encode(
        canonical_names,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    scores = cosine_similarity(embeddings, candidate_embeddings)
    expected = [normalize_name(value) for value in examples["canonical_name"]]
    ranks = []
    for row_scores, expected_name in zip(scores, expected):
        order = np.argsort(-row_scores)
        expected_index = next(
            (index for index, name in enumerate(canonical_names) if normalize_name(name) == expected_name),
            None,
        )
        ranks.append(len(canonical_names) + 1 if expected_index is None else int(np.where(order == expected_index)[0][0]) + 1)
    ranks = np.asarray(ranks)
    return {
        "samples": float(len(ranks)),
        "top_1_accuracy": float(np.mean(ranks <= 1)),
        "top_3_recall": float(np.mean(ranks <= 3)),
        "median_rank": float(np.median(ranks)),
    }

def evaluation_trigrams(value: str) -> set[str]:
    return {value} if len(value) < 3 else {value[index : index + 3] for index in range(len(value) - 2)}

def evaluation_score(raw_name: str, candidate_name: str) -> float:
    raw, candidate = normalize_name(raw_name), normalize_name(candidate_name)
    raw_tri, candidate_tri = evaluation_trigrams(raw), evaluation_trigrams(candidate)
    tri = len(raw_tri & candidate_tri) / len(raw_tri | candidate_tri) if raw_tri | candidate_tri else 1.0
    prefix = 0
    for left, right in zip(raw, candidate):
        if left != right:
            break
        prefix += 1
    prefix = prefix / max(len(raw), len(candidate), 1)
    left_digits = "".join(re.findall(r"\d+", raw_name))
    right_digits = "".join(re.findall(r"\d+", candidate_name))
    digit = 1.0 if left_digits and left_digits == right_digits else 0.5 if left_digits and right_digits and (left_digits.startswith(right_digits) or right_digits.startswith(left_digits)) else 0.0
    return 0.60 * tri + 0.25 * prefix + 0.15 * digit

def deterministic_retrieval_metrics(examples: pd.DataFrame) -> dict[str, float]:
    ranks = []
    for row in examples.itertuples(index=False):
        scores = np.asarray([
            evaluation_score(row.raw_name, candidate)
            for candidate in canonical_names
        ])
        order = np.argsort(-scores)
        expected_name = normalize_name(row.canonical_name)
        expected_index = next(
            (index for index, name in enumerate(canonical_names) if normalize_name(name) == expected_name),
            None,
        )
        ranks.append(len(canonical_names) + 1 if expected_index is None else int(np.where(order == expected_index)[0][0]) + 1)
    ranks = np.asarray(ranks)
    return {
        "samples": float(len(ranks)),
        "top_1_accuracy": float(np.mean(ranks <= 1)),
        "top_3_recall": float(np.mean(ranks <= 3)),
        "median_rank": float(np.median(ranks)),
    }

metrics = {
    "deterministic_fuzzy_baseline": deterministic_retrieval_metrics(validation_pairs),
    "pretrained_encoder": retrieval_metrics(base_encoder, validation_pairs),
    "fine_tuned_encoder": retrieval_metrics(encoder, validation_pairs),
}
metrics_table = pd.DataFrame(metrics).T
display(metrics_table)
print("Catatan: metrik harus dibaca bersama jumlah pasangan validasi dan split berdasarkan canonical unit.")

,samples,top_1_accuracy,top_3_recall,median_rank
deterministic_fuzzy_baseline,12.0,1.000000,1.000000,1.0
pretrained_encoder,12.0,0.666667,0.833333,1.0
fine_tuned_encoder,12.0,0.666667,0.833333,1.0


Catatan: metrik harus dibaca bersama jumlah pasangan validasi dan split berdasarkan canonical unit.


## 6. Hybrid inference dengan guardrail deterministik

Ranking ML digabungkan dengan komponen skor pada spesifikasi alias. Auto-map membutuhkan skor deterministik tinggi, margin yang cukup, dan activity yang kompatibel. Model tidak boleh mengesampingkan kasus ambigu.

In [7]:
AUTO_MAP_MIN_SCORE = 0.95
AUTO_MAP_MIN_MARGIN = 0.10
REVIEW_MIN_SCORE = 0.70
FORCE_REVIEW_NORMALIZED = {normalize_name("DREDGER 12/1")}

def trigrams(value: str) -> set[str]:
    if len(value) < 3:
        return {value}
    return {value[index : index + 3] for index in range(len(value) - 2)}

def trigram_similarity(left: str, right: str) -> float:
    a, b = trigrams(left), trigrams(right)
    return len(a & b) / len(a | b) if a | b else 1.0

def prefix_ratio(left: str, right: str) -> float:
    common = 0
    for left_char, right_char in zip(left, right):
        if left_char != right_char:
            break
        common += 1
    return common / max(len(left), len(right), 1)

def digit_sequence_match(left: str, right: str) -> float:
    left_digits = "".join(re.findall(r"\d+", left))
    right_digits = "".join(re.findall(r"\d+", right))
    if not left_digits or not right_digits:
        return 0.0
    if left_digits == right_digits:
        return 1.0
    if left_digits.startswith(right_digits) or right_digits.startswith(left_digits):
        return 0.5
    return 0.0

def deterministic_score(raw_name: str, candidate_name: str) -> float:
    raw, candidate = normalize_name(raw_name), normalize_name(candidate_name)
    return (
        0.60 * trigram_similarity(raw, candidate)
        + 0.25 * prefix_ratio(raw, candidate)
        + 0.15 * digit_sequence_match(raw_name, candidate_name)
    )

def rank_candidates(raw_name: str, declared_activity: str | None = None, top_k: int = 5) -> pd.DataFrame:
    query_embedding = encoder.encode([raw_name], convert_to_numpy=True, normalize_embeddings=True)
    ml_scores = cosine_similarity(query_embedding, canonical_embeddings)[0]
    ranked = catalog.copy()
    ranked["ml_score"] = ml_scores
    ranked["deterministic_score"] = ranked["canonical_name"].map(
        lambda candidate: deterministic_score(raw_name, candidate)
    )
    declared = activity_key(declared_activity)
    ranked["activity_match"] = ranked["activity"].map(
        lambda activity: declared is None or activity is None or activity == declared
    )
    ranked = ranked.sort_values(
        ["ml_score", "deterministic_score"], ascending=False
    ).head(top_k).reset_index(drop=True)
    return ranked

def resolve_alias(raw_name: str, declared_activity: str | None = None, top_k: int = 5) -> dict[str, object]:
    normalized = normalize_name(raw_name)
    if normalized in FORCE_REVIEW_NORMALIZED:
        return {
            "raw_name": raw_name,
            "normalized_name": normalized,
            "decision": "review",
            "reason": "domain_truncation_requires_review",
            "canonical_name": None,
            "candidates": rank_candidates(raw_name, declared_activity, top_k).to_dict(orient="records"),
        }
    exact = catalog[catalog["normalized_name"] == normalized]
    declared = activity_key(declared_activity)
    if not exact.empty and (
        declared is None
        or exact.iloc[0]["activity"] is None
        or exact.iloc[0]["activity"] == declared
    ):
        candidate = exact.iloc[0]
        return {
            "raw_name": raw_name,
            "normalized_name": normalized,
            "decision": "auto_map",
            "reason": "exact_normalized_match",
            "canonical_name": candidate["canonical_name"],
            "candidates": rank_candidates(raw_name, declared_activity, top_k).to_dict(orient="records"),
        }

    ranked = rank_candidates(raw_name, declared_activity, top_k)
    if ranked.empty:
        return {"raw_name": raw_name, "normalized_name": normalized, "decision": "unknown", "candidates": []}
    top = ranked.iloc[0]
    second_score = float(ranked.iloc[1]["deterministic_score"]) if len(ranked) > 1 else 0.0
    margin = float(top["deterministic_score"]) - second_score
    activity_ok = bool(top["activity_match"])
    if (
        activity_ok
        and float(top["deterministic_score"]) >= AUTO_MAP_MIN_SCORE
        and margin >= AUTO_MAP_MIN_MARGIN
    ):
        decision = "auto_map"
        reason = "ml_ranked_and_deterministic_guardrails_passed"
    elif float(top["deterministic_score"]) >= REVIEW_MIN_SCORE or not activity_ok:
        decision = "review"
        reason = "ambiguous_or_guardrail_review"
    else:
        decision = "unknown"
        reason = "no_confident_candidate"
    return {
        "raw_name": raw_name,
        "normalized_name": normalized,
        "decision": decision,
        "reason": reason,
        "canonical_name": top["canonical_name"] if decision == "auto_map" else None,
        "candidates": ranked.to_dict(orient="records"),
    }

## 7. Golden cases dan demonstrasi

Kasus truncation dan nama yang belum aman harus masuk review, bukan dipaksa menjadi mapping.

In [8]:
golden_cases = [
    {"raw_name": "D85ESS-2", "expected": "D85ESS2", "expected_decision": "auto_map"},
    {"raw_name": "RF-85MW", "expected": "RF85MW", "expected_decision": "auto_map"},
    {"raw_name": "DREDGER 12/1", "expected": None, "expected_decision": "review"},
    {"raw_name": "HD7857OTD", "expected": None, "expected_decision": "review_or_unknown"},
]
golden_results = []
for case in golden_cases:
    result = resolve_alias(case["raw_name"])
    golden_results.append(
        {
            "raw_name": case["raw_name"],
            "expected": case["expected"],
            "predicted": result.get("canonical_name"),
            "decision": result["decision"],
            "reason": result["reason"],
        }
    )
golden_results_df = pd.DataFrame(golden_results)
display(golden_results_df)

assert golden_results_df.loc[0, "predicted"] == "D85ESS2"
assert golden_results_df.loc[1, "predicted"] == "RF85MW"
assert golden_results_df.loc[2, "decision"] == "review"
assert golden_results_df.loc[3, "decision"] in {"review", "unknown"}
print("Golden cases passed")

demo_queries = [
    ("PUMP, MULTIFLO, MF420EXHV, CAT C27", "dewatering"),
    ("PC 2000_Mud", "loading"),
    ("DREDGER 12/1", "dewatering"),
]
for query, activity in demo_queries:
    result = resolve_alias(query, activity)
    print(query, "→", result["decision"], result.get("canonical_name"), result["reason"])

,raw_name,expected,predicted,decision,reason
0,D85ESS-2,D85ESS2,D85ESS2,auto_map,exact_normalized_match
1,RF-85MW,RF85MW,RF85MW,auto_map,exact_normalized_match
2,DREDGER 12/1,None,None,review,domain_truncation_requires_review
3,HD7857OTD,None,None,unknown,no_confident_candidate


Golden cases passed
PUMP, MULTIFLO, MF420EXHV, CAT C27 → review None ambiguous_or_guardrail_review
PC 2000_Mud → unknown None no_confident_candidate
DREDGER 12/1 → review None domain_truncation_requires_review


## 8. Simpan artefak model dan evaluasi

Artefak ini adalah output eksperimen lokal. Backend tetap harus menerapkan validasi activity, review queue, dan audit trail sebelum menerima mapping.

In [9]:
artifact_dir = server_root / "ml_artifacts" / "alias_resolution"
artifact_dir.mkdir(parents=True, exist_ok=True)
encoder.save(str(artifact_dir / "encoder"))
joblib.dump(
    {
        "canonical_catalog": catalog,
        "canonical_embeddings": canonical_embeddings,
        "model_name": MODEL_NAME,
        "normalization": "NFKC, trim, uppercase, collapse whitespace, remove separators",
        "thresholds": {
            "auto_map_min_score": AUTO_MAP_MIN_SCORE,
            "auto_map_min_margin": AUTO_MAP_MIN_MARGIN,
            "review_min_score": REVIEW_MIN_SCORE,
        },
    },
    artifact_dir / "candidate_index.joblib",
)

evaluation = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "model_name": MODEL_NAME,
    "seed": SEED,
    "sources": {
        "workbook": str(workbook_path.relative_to(project_root)),
        "reference_doc": str(reference_doc_path.relative_to(project_root)),
        "alias_doc": str(alias_doc_path.relative_to(project_root)),
    },
    "catalog_size": len(catalog),
    "training_pairs": len(train_pairs),
    "validation_pairs": len(validation_pairs),
    "metrics": metrics,
    "golden_cases": golden_results,
    "guardrail": "ML hanya meranking kandidat; calculation engine tetap sumber angka",
}
(artifact_dir / "evaluation.json").write_text(
    json.dumps(evaluation, ensure_ascii=False, indent=2), encoding="utf-8"
)
print(f"Saved artifacts to {artifact_dir}")

Saved artifacts to /content/ml_artifacts/alias_resolution


In [10]:
import shutil
from pathlib import Path

download_source = Path("/content/ml_artifacts/alias_resolution")
download_base = Path("/content/alias_resolution_artifacts")
if not download_source.exists():
    raise FileNotFoundError(download_source)
archive_path = Path(shutil.make_archive(str(download_base), "zip", download_source.parent, download_source.name))
print(f"Created archive: {archive_path}")
try:
    from google.colab import files
    files.download(str(archive_path))
except ImportError:
    print("Automatic download is available only in Google Colab.")

Created archive: /content/alias_resolution_artifacts.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>